# Publications markdown generator for academicpages

Takes a set of bibtex of publications and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). 

The core python code is also in `pubsFromBibs.py`. 
Run either from the `markdown_generator` folder after replacing updating the publist dictionary with:
* bib file names
* specific venue keys based on your bib file preferences
* any specific pre-text for specific files
* Collection Name (future feature)

TODO: Make this work with other databases of citations, 
TODO: Merge this with the existing TSV parsing solution

In [1]:
from pybtex.database.input import bibtex
import pybtex.database.input.bibtex 
from time import strptime
import string
import html
import os
import re

In [2]:
#todo: incorporate different collection types rather than a catch all publications, requires other changes to template
publist = {
    "proceeding": {
        "file" : "proceedings.bib",
        "venuekey": "booktitle",
        "venue-pretext": "In the proceedings of ",
        "collection" : {"name":"publications",
                        "permalink":"/publication/"}
        
    },
    "journal":{
        "file": "pubs.bib",
        "venuekey" : "journal",
        "venue-pretext" : "",
        "collection" : {"name":"publications",
                        "permalink":"/publication/"}
    } 
}

In [3]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    """Produce entities within text."""
    return "".join(html_escape_table.get(c,c) for c in text)

In [18]:
myName = "Shivam Agarwal"
for pubsource in publist:
    parser = bibtex.Parser()
    bibdata = parser.parse_file(publist[pubsource]["file"])

    #loop through the individual references in a given bibtex file
    for bib_id in bibdata.entries:
        #reset default date
        pub_year = "1900"
        pub_month = "01"
        pub_day = "01"
        
        b = bibdata.entries[bib_id].fields
        
        try:
            pub_year = f'{b["year"]}'

            #todo: this hack for month and day needs some cleanup
            if "month" in b.keys(): 
                if(len(b["month"])<3):
                    pub_month = "0"+b["month"]
                    pub_month = pub_month[-2:]
                elif(b["month"] not in range(12)):
                    tmnth = strptime(b["month"][:3],'%b').tm_mon   
                    pub_month = "{:02d}".format(tmnth) 
                else:
                    pub_month = str(b["month"])
            if "day" in b.keys(): 
                pub_day = str(b["day"])

                
            pub_date = pub_year+"-"+pub_month+"-"+pub_day
            
            #strip out {} as needed (some bibtex entries that maintain formatting)
            clean_title = b["title"].replace("{", "").replace("}","").replace("\\","").replace(" ","-")    

            url_slug = re.sub("\\[.*\\]|[^a-zA-Z0-9_-]", "", clean_title)
            url_slug = url_slug.replace("--","-")

            md_filename = (str(pub_date) + "-" + url_slug + ".md").replace("--","-")
            html_filename = (str(pub_date) + "-" + url_slug).replace("--","-")

            #Build Citation from text
            citation = ""

            

            
            ## YAML variables
            
            md = "---\ntitle: \""   + paperTitle +  '"\n'
            
            md += """collection: """ +  publist[pubsource]["collection"]["name"]

            md += """\npermalink: """ + publist[pubsource]["collection"]["permalink"]  + html_filename
            
            note = False
            if "note" in b.keys():
                if len(str(b["note"])) > 5:
                    md += "\nexcerpt: '" + html_escape(b["note"]) + "'"
                    note = True

            md += "\ndate: " + str(pub_date) 

            md += "\nvenue: '" + html_escape(venue) + "'"
            
            url = False
            if "paperurl" in b.keys():
                if len(str(b["paperurl"])) > 5:
                    md += "\npaperurl: '" + b["paperurl"] + "'"
                    url = True
                    
            image = False
            if "image" in b.keys():
                if len(str(b["image"])) > 5:
                    md += "\nimage: '" + b["image"] + "'"
                    image = True
            code = False
            if "code" in b.keys():
                md += "\ncode: '" + b["code"] + "'"
                code = True
                
            demo = False
            if "demo" in b.keys():
                md += "\ndemo: '" + b["demo"] + "'"
                demo = True
                
            doi = False
            if "doi" in b.keys():
                md += "\ndoi: '" + b["doi"] + "'"
                doi = True
                
            doiurl = False
            if "doi" in b.keys():
                md += "\ndoiurl: 'https://dx.doi.org/" + b["doi"] + "'"
                doiurl = True
                    
            abstract = False
            if "abstract" in b.keys():
                md += "\nabstract: '" + b["abstract"] + "'"
                abstract = True
            
            video = False
            if "video" in b.keys():
                md += "\nvideo: '" + b["video"] + "'"
                video = True
                
            youtubeid = False
            if "video" in b.keys():
                vidsplit = b["video"].split("?v=")
                md += "\nyoutubeid: '" + vidsplit[1] + "'"
                youtubeid = True
            
            #citation authors - todo - add highlighting for primary author?
            for author in bibdata.entries[bib_id].persons["author"]:
#                 To make my name in bold while listing the publications
                authorFullName = author.first_names[0]+" "+author.last_names[0]
                if authorFullName == myName:
                    authorFullName = "<u>"+authorFullName+"</u>"
                citation = citation+" "+authorFullName+", "

            #citation title
            paperTitle = html_escape(b["title"].replace("{", "").replace("}","").replace("\\",""))
            citation = citation + "<i>\"" + paperTitle + "\"</i>"

            #add venue logic depending on citation type
            venue = publist[pubsource]["venue-pretext"]+b[publist[pubsource]["venuekey"]].replace("{", "").replace("}","").replace("\\","")

            citation = citation + " " + html_escape(venue)
            citation = citation + ", " + pub_year + ". "
#             if doi:
#                 citation = citation + b["doi"] 

            md += "\ncitation: '" + html_escape(citation) + "'"

            md += "\n---"
            
            

            
            ## Markdown description for individual page
            if note:
                md += "\n" + html_escape(b["note"]) + "\n"
            
#             if doi: 
#                 md += "\n[Access paper here](" + b["doi"] + "){:target=\"_blank\"}\n" 
#             else if url:
#                 md += "\n[Access paper here](" + b["url"] + "){:target=\"_blank\"}\n" 
#             else:
#                 md += "\nUse [Google Scholar](https://scholar.google.com/scholar?q="+html.escape(clean_title.replace("-","+"))+"){:target=\"_blank\"} for full citation"

            md_filename = os.path.basename(md_filename)

            with open("../_publications/" + md_filename, 'w', encoding='utf-8') as f:
                f.write(md)
            print(f'SUCESSFULLY PARSED {bib_id}: \"', b["title"][:60],"..."*(len(b['title'])>60),"\"")
        # field may not exist for a reference
        except KeyError as e:
            print(f'WARNING Missing Expected Field {e} from entry {bib_id}: \"', b["title"][:30],"..."*(len(b['title'])>30),"\"")
            continue


SUCESSFULLY PARSED Agarwal2017Nodetrix: " NodeTrix-Multiplex: Visual Analytics of Multiplex Small Worl ... "
SUCESSFULLY PARSED Nair2017Nodetrix: " NodeTrix-CommunityHierarchy: Techniques for Finding Hierarch ... "
SUCESSFULLY PARSED Lukose2018Design: " Design Study for Creating Pathfinder: A Visualization Tool f ... "
SUCESSFULLY PARSED Agarwal2018Computer: " Computer-supported Interactive Assignment of Keywords for Li ... "
SUCESSFULLY PARSED Agarwal2020Visualizing: " Visualizing AI Playtesting Data of 2D Side-scrolling Games  "
SUCESSFULLY PARSED Agarwal2020Design: " A Design and Application Space for Visualizing User Sessions ... "
SUCESSFULLY PARSED Agarwal2020VisualizingSets: " Visualizing Sets and Changes in Membership Using Layered Set ... "
SUCESSFULLY PARSED Agarwal2020How: " How Visualization PhD Students Cope with Paper Rejections  "
SUCESSFULLY PARSED Krause2021Visual: " Visual Comparison of Multi-label Classification Results  "
SUCESSFULLY PARSED Latif2021Visually: " Visu